# 01 · Comprensión del negocio (CRISP-DM — Fase 1)

## Alcance del estudio y premisa científica

La predicción determinista de terremotos —la especificación anticipada de
fecha, hora, lugar y magnitud de un evento concreto— no resulta alcanzable
con el estado actual del conocimiento sismológico, conforme lo sostiene el
**USGS** y la literatura especializada; cualquier producto que afirme esa
capacidad carece de sustento científico. Este estudio, por tanto, no persigue
dicho objetivo: se inscribe en el paradigma del **peligro sísmico
probabilístico** y en la estadística de secuencias de réplicas, cuyos
productos analíticos se enumeran a continuación.

| Componente | Pregunta que responde | Método |
|---|---|---|
| **Peligro sísmico relativo** | ¿Dónde se han concentrado históricamente la sismicidad y la energía liberada? | Estadística de catálogo (frecuencia + energía, escala Gutenberg-Richter) |
| **Exposición poblacional** | ¿Cuánta población reside en cada cantón y a qué densidad? | Censo de Población y Vivienda 2022 (INEC) |
| **Índice de riesgo relativo** | ¿En qué cantones la confluencia de peligro y exposición prioriza la inversión en construcción sismorresistente? | Peligro × exposición, a nivel cantonal |
| **Pronóstico de la tasa de réplicas** | Tras un sismo principal, ¿cuál es la tasa diaria esperada de réplicas y cuál su ley de decaimiento? | Ley de Omori-Utsu sobre la secuencia de Pedernales 2016 (M7.8) |

Corresponde precisar que el pronóstico de réplicas estima la **tasa
esperada** de la secuencia (sismos/día) como estadístico de un proceso
estocástico, y no la ocurrencia de eventos individuales. Esta distinción
delimita la validez del modelo y se retoma en los cuadernos 05 y 06.

## Contexto del problema

Ecuador se sitúa sobre el margen convergente donde la placa Nazca se subduce bajo
la placa Sudamericana. Esta configuración tectónica produce:

- **Sismos de subducción** (interplaca): los de mayor magnitud y tsunami potencial
  frente a la costa: 1906 M~8.8, 1942 M7.9, 1958 M7.8, 1979 M8.2 y **2016 M7.8 Pedernales**.
- **Sismos corticales** (intraplaca): más superficiales y cercanos a las ciudades,
  como el de **Ambato/Pelileo 1949** o el de **Pujilí 2022**, sentido fuertemente en Quito.
- **Sismicidad profunda** (>100 km) bajo la Amazonía occidental.

Tras el terremoto de Pedernales (16-abril-2016, M7.8, cientos de fallecidos y
pérdidas millonarias), la pregunta de política pública relevante no es "cuándo
temblará" sino **"dónde concentrar la inversión en reducción de riesgo"**: norma
sismorresistente (NEC), reforzamiento de edificaciones, hospitales y escuelas,
y preparación ante réplicas.

## Objetivos del proyecto (criterios de éxito)

1. Construir un **índice de riesgo sísmico relativo por cantón** reproducible,
   basado únicamente en fuentes oficiales, que priorice territorios.
2. Ajustar y **validar un modelo de decaimiento de réplicas** (Omori-Utsu) con
   datos reales de la secuencia de Pedernales 2016.
3. Contrastar cualitativamente el índice contra la **zonificación sísmica de la
   Norma Ecuatoriana de la Construcción (NEC-SE-DS)**.
4. (Exploratorio) Revisar si el catálogo muestra **actividad anómala cerca de
   grandes embalses** (Mazar, Coca Codo Sinclair, Paute), sin asumir causalidad.
4. Documentar todo el proceso con **CRISP-DM** y código comentado en español.

## Fuentes de datos oficiales

| Dato | Fuente | Detalle |
|---|---|---|
| Catálogo sísmico 1983–2026 | **IG-EPN** (oficial, descarga bajo solicitud) y servicio **FDSN/USGS** (oficial, abierto) | Eventos M≥3/4: fecha, hora, magnitud, profundidad, lat/lon. Usamos USGS-FDSN por reproducibilidad programática; el IG-EPN publica además reportes por evento |
| Peligros por territorio 2010–2022 | **SNGRE** (datosabiertos.gob.ec) | Eventos peligrosos por provincia/cantón/parroquia |
| Población por cantón | **INEC — Censo de Población y Vivienda 2022** | Tabulado 1.1 (estructura poblacional) |
| Límites cantonales | **IGM/CONALI** vía geoBoundaries (gbOpen, basado en INEC/OCHA) | 224 cantones, CC-BY 3.0 IGO |
| Caso de réplicas | **Terremoto de Pedernales, 16-abr-2016, M7.8** | IG-EPN/USGS |
| Normativa | **NEC-SE-DS** (peligro sísmico, 2015) | Zonificación para contraste cualitativo |

> **Nota sobre el período:** el alcance temporal es **1983 – agosto 2026** (43 años
> de catálogo) y el nivel de agregación es **cantonal** (224 unidades). Ambos
> parámetros son configurables en `src/fetch_data.py` y `src/data_prep.py`.

## Plan CRISP-DM del repositorio

| Fase | Notebook |
|---|---|
| 1. Comprensión del negocio | `01_business_understanding.ipynb` |
| 2. Comprensión de los datos | `02_data_understanding.ipynb` |
| 3. Preparación de los datos | `03_data_preparation.ipynb` |
| 4. Modelado (peligro × exposición) | `04_modeling_peligro_exposicion.ipynb` |
| 5. Modelado (réplicas) | `05_modeling_replicas.ipynb` |
| 6. Evaluación | `06_evaluation.ipynb` |

In [1]:
# --- Verificación del entorno y datos disponibles ---
import os
import pathlib

In [2]:
RAIZ = pathlib.Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
os.chdir(RAIZ)

In [3]:
print(f"Raíz del proyecto: {RAIZ}")
print("\nArchivos en data/raw/ (deben existir 5):")
for f in sorted(os.listdir("data/raw")):
    kb = os.path.getsize(os.path.join("data/raw", f)) / 1024
    print(f"  - {f} ({kb:,.0f} KB)")

Raíz del proyecto: C:\Users\Jordan\.zcode\workspace\default\riesgo-sismico-ecuador

Archivos en data/raw/ (deben existir 5):
  - cantones_ecuador.geojson (2,995 KB)
  - catalogo_sismico_ecuador_1983_2026.csv (438 KB)
  - catalogo_sismico_ecuador_M3_1983_2026.csv (471 KB)
  - censo2022_inec.zip (364 KB)
  - censo2022_poblacion_canton.csv (42 KB)


In [4]:
assert len(os.listdir("data/raw")) >= 4, "Ejecuta primero: python src/fetch_data.py"
print("\n[ok] Datos crudos presentes. Continúa con 02_data_understanding.ipynb")


[ok] Datos crudos presentes. Continúa con 02_data_understanding.ipynb
